# DINOv2 Baseline Diagnostic & Ablation Study Evaluation

This notebook presents the diagnostic investigations and ablation evaluations for our **DINOv2 Baseline** on the DogReID-1553 benchmark:

1. **Subject Masking & Crop Quality**: Disentangling dog biometric features from background context and evaluating detection crop precision.
2. **Number of Frames (Temporal Scaling)**: Evaluating inference scaling across 1, 4, 8 (baseline), 16, 32, and 64 frames with dynamic temporal attention pooling.
3. **Feature Pooling & Aggregation**: Comparing learnable temporal self-attention against mean/max pooling across frames, as well as spatial token pooling within the backbone.
4. **Gallery Size Sensitivity Analysis**: Evaluating retrieval robustness as the gallery expands from 1 video per identity (38.9% / 777 videos) to 100% (1,998 videos).
5. **Failure Case Analysis**: Framework for qualitative inspection of retrieval errors, hard negatives, and viewpoint shifts.

All metrics are reported with **95% Bootstrap Confidence Intervals** (m = 100 iterations).

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add root directory to sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR if (CURRENT_DIR / "evaluation").exists() else CURRENT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
if str(ROOT_DIR / "evaluation") not in sys.path:
    sys.path.append(str(ROOT_DIR / "evaluation"))

from evaluation.evaluation_utils import bootstrap_from_csv
print("Loaded evaluation_utils successfully!")
print(f"Project root: {ROOT_DIR}")

Loaded evaluation_utils successfully!
Project root: /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553


## 1. Subject Masking & Crop Quality Diagnostic Evaluation

Diagnostic evaluation to determine how much the model relies on **dog appearance vs. background contextual clues**, and the effect of **bounding box localization noise**:
- **V2V YOLO-Masked**: The dog region is masked out (blacked out) across all video frames to evaluate background bias.
- **I2V Standard (YOLO Crop)**: Automated YOLO detection bounding boxes on single image query vs. video gallery.
- **I2V GT Crop**: Precise ground-truth human-annotated bounding box on image query vs. video gallery.
- **I2V Masked Queries**: Subject masked out in query image (YOLO-masked vs. GT-masked).

In [2]:
# =====================================================================
# 1. Subject Masking & Crop Quality Diagnostic Table
# =====================================================================
DIAG_DIR = ROOT_DIR / "evaluation" / "csvs_diagnostics"
CLOSED_DIR = ROOT_DIR / "evaluation" / "csvs_closed_results"
BASE_CKPT = ROOT_DIR / "checkpoints" / "baseline" / "closed_baseline_ep50" / "latest_model.pth"
BASE_MTIME = BASE_CKPT.stat().st_mtime if BASE_CKPT.exists() else 0

def get_row(name, path, fallback_path=None, ref_r1=None, ref_map=None):
    p = path if path.exists() and path.stat().st_size > 0 else (fallback_path if fallback_path and fallback_path.exists() and fallback_path.stat().st_size > 0 else None)
    if not p or p.stat().st_mtime < BASE_MTIME:
        return {"Setting": name, "Rank-1 (%) [95% CI]": "Pending...", "Rank-5 (%) [95% CI]": "Pending...", "mAP (%) [95% CI]": "Pending...", "Δ (R-1 / mAP)": "-", "Status": "In Progress"}
    res = bootstrap_from_csv(str(p), m=100, mode="closed")
    m_mean = res.get("mAP_boot_mean", 0.0) * 100
    m_std = res.get("mAP_std", 0.0) * 100
    m_ci = res.get("mAP_ci", (0.0, 0.0))
    cmc_boot = res.get("cmc_boot_mean", [0.0]*10)
    cmc_std = res.get("cmc_std", np.zeros(10))
    cmc_l = res.get("cmc_ci_lower", cmc_boot)
    cmc_u = res.get("cmc_ci_upper", cmc_boot)
    cmc = cmc_boot[0] * 100
    r1_std = cmc_std[0] * 100 if len(cmc_std) > 0 else 0.0
    cmc_ci = (cmc_l[0] * 100, cmc_u[0] * 100)
    r5 = cmc_boot[4] * 100 if len(cmc_boot) >= 5 else 0.0
    r5_std = cmc_std[4] * 100 if len(cmc_std) >= 5 else 0.0
    r5_ci = (cmc_l[4] * 100, cmc_u[4] * 100) if len(cmc_l) >= 5 else (r5, r5)
    delta = "Reference" if ref_r1 is None else f"{(cmc - ref_r1):+.1f} / {(m_mean - ref_map):+.1f}"
    return {
        "Setting": name,
        "Rank-1 (%) [95% CI]": f"{cmc:.1f} ± {r1_std:.1f} [{cmc_ci[0]:.1f}-{cmc_ci[1]:.1f}]",
        "Rank-5 (%) [95% CI]": f"{r5:.1f} ± {r5_std:.1f} [{r5_ci[0]:.1f}-{r5_ci[1]:.1f}]",
        "mAP (%) [95% CI]": f"{m_mean:.1f} ± {m_std:.1f} [{m_ci[0]*100:.1f}-{m_ci[1]*100:.1f}]",
        "Δ (R-1 / mAP)": delta,
        "Status": "Verified",
        "_r1": cmc, "_r5": r5, "_map": m_mean
    }

base_v2v = get_row("V2V Standard (Unmasked)", CLOSED_DIR / "baseline_vid2vid" / "closed_dist_matrix.csv", ROOT_DIR / "evaluation/csvs_ablations/baseline/closed_dist_matrix.csv")
v2v_r1, v2v_map = base_v2v.get("_r1"), base_v2v.get("_map")

base_i2v = get_row("I2V Standard (YOLO Crop, Unmasked)", DIAG_DIR / "baseline_img2vid_unmasked_yolo" / "closed_dist_matrix.csv", CLOSED_DIR / "baseline_img2vid" / "closed_dist_matrix.csv")
i2v_r1, i2v_map = base_i2v.get("_r1"), base_i2v.get("_map")

masking_rows = [
    base_v2v,
    get_row("V2V YOLO-Masked (Background-Only)", DIAG_DIR / "baseline_vid2vid_masked" / "closed_dist_matrix.csv", ref_r1=v2v_r1, ref_map=v2v_map),
    base_i2v,
    get_row("I2V GT Crop (Unmasked)", DIAG_DIR / "baseline_img2vid_unmasked_gt" / "closed_dist_matrix.csv", ref_r1=i2v_r1, ref_map=i2v_map),
    get_row("I2V YOLO-Masked Query", DIAG_DIR / "baseline_img2vid_masked_yolo" / "closed_dist_matrix.csv", ref_r1=i2v_r1, ref_map=i2v_map),
    get_row("I2V GT-Masked Query", DIAG_DIR / "baseline_img2vid_masked_gt" / "closed_dist_matrix.csv", ref_r1=i2v_r1, ref_map=i2v_map),
]
df_masking = pd.DataFrame(masking_rows)[["Setting", "Rank-1 (%) [95% CI]", "Rank-5 (%) [95% CI]", "mAP (%) [95% CI]", "Δ (R-1 / mAP)", "Status"]]

print("=== Subject Masking & Crop Quality Diagnostic ===")
display(df_masking)


-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
=== Subject Masking & Crop Quality Diagnostic ===
                              Setting  ...    Status
0             V2V Standard (Unmasked)  ...  Verified
1   V2V YOLO-Masked (Background-Only)  ...  Verified
2  I2V Standard (YOLO Crop, Unmasked)  ...  Verified
3     

## 2. Number of Frames (Temporal Frame Scaling)

Evaluation of retrieval accuracy across varying video temporal sampling rates (1, 4, 8, 16, 32, and 64 frames per tracklet clip).
Our baseline model is trained on 8 frames with dynamic self-attention pooling.

In [3]:
# =====================================================================
# 2. Number of Frames (Temporal Scaling) Table
# =====================================================================
f8_r1, f8_map = v2v_r1, v2v_map
frame_rows = [
    get_row("1 Frame", DIAG_DIR / "baseline_vid2vid_len1" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
    get_row("4 Frames", DIAG_DIR / "baseline_vid2vid_len4" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
    {"Setting": "8 Frames", "Rank-1 (%) [95% CI]": base_v2v["Rank-1 (%) [95% CI]"], "Rank-5 (%) [95% CI]": base_v2v["Rank-5 (%) [95% CI]"], "mAP (%) [95% CI]": base_v2v["mAP (%) [95% CI]"], "Δ (R-1 / mAP)": "Reference", "Status": "Verified", "_r1": f8_r1, "_r5": base_v2v.get("_r5"), "_map": f8_map},
    get_row("16 Frames", DIAG_DIR / "baseline_vid2vid_len16" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
    get_row("32 Frames", DIAG_DIR / "baseline_vid2vid_len32" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
    get_row("64 Frames", DIAG_DIR / "baseline_vid2vid_len64" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
]
df_frames = pd.DataFrame(frame_rows)[["Setting", "Rank-1 (%) [95% CI]", "Rank-5 (%) [95% CI]", "mAP (%) [95% CI]", "Δ (R-1 / mAP)", "Status"]]

print("=== Number of Frames Inference Diagnostic ===")
display(df_frames)


-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
=== Number of Frames Inference Diagnostic ===
     Setting     Rank-1 (%) [95% CI]  ...  Δ (R-1 / mAP)    Status
0    1 Frame  43.5 ± 1.3 [41.2-46.1]  ...  -18.7 / -16.3  Verified
1   4 Frames  58.2 ± 1.5 [55.7-61.1]  ...    -4.0 / -2.9  Verified
2   8 Frames  62.2 ± 1.5 [59.4-64.8]  ...      Reference  Verified
3  16 Frames  62.3 ± 1.5 [59.9-66.0]  ...    +0.2 / +1.0  Verified
4  32 Fra

### Temporal Frame Scaling Curve (Rank-1 & mAP with 95% Confidence Intervals)

This plot visualizes how retrieval performance scales as the number of sampled video frames increases from 1 to 64 frames per tracklet clip.

In [4]:
# =====================================================================
# Plot: Performance vs. Number of Frames (with 95% Bootstrap CIs)
# =====================================================================
frame_plot_defs = [
    {"frames": 1, "label": "1", "path": DIAG_DIR / "baseline_vid2vid_len1" / "closed_dist_matrix.csv"},
    {"frames": 4, "label": "4", "path": DIAG_DIR / "baseline_vid2vid_len4" / "closed_dist_matrix.csv"},
    {"frames": 8, "label": "8 (Baseline)", "path": CLOSED_DIR / "baseline_vid2vid" / "closed_dist_matrix.csv"},
    {"frames": 16, "label": "16", "path": DIAG_DIR / "baseline_vid2vid_len16" / "closed_dist_matrix.csv"},
    {"frames": 32, "label": "32", "path": DIAG_DIR / "baseline_vid2vid_len32" / "closed_dist_matrix.csv"},
    {"frames": 64, "label": "64", "path": DIAG_DIR / "baseline_vid2vid_len64" / "closed_dist_matrix.csv"},
]

frames_eval = []
for item in frame_plot_defs:
    p = item["path"]
    if p.exists() and p.stat().st_size > 0 and p.stat().st_mtime >= BASE_MTIME:
        res = bootstrap_from_csv(str(p), m=100, mode="closed")
        r1 = res["cmc_boot_mean"][0] * 100
        r1_l = res["cmc_ci_lower"][0] * 100
        r1_u = res["cmc_ci_upper"][0] * 100
        m_val = res["mAP_boot_mean"] * 100
        m_l = res["mAP_ci"][0] * 100
        m_u = res["mAP_ci"][1] * 100
        frames_eval.append({
            "frames": item["frames"],
            "label": item["label"],
            "r1": r1,
            "r1_err": [[max(0.0, r1 - r1_l)], [max(0.0, r1_u - r1)]],
            "map": m_val,
            "map_err": [[max(0.0, m_val - m_l)], [max(0.0, m_u - m_val)]]
        })

if len(frames_eval) >= 2:
    fig, ax = plt.subplots(figsize=(9, 5.5), dpi=200)
    x = [item["frames"] for item in frames_eval]
    r1_y = [item["r1"] for item in frames_eval]
    r1_errs = np.array([item["r1_err"] for item in frames_eval]).squeeze().T
    if r1_errs.ndim == 1:
        r1_errs = r1_errs.reshape(2, 1)

    map_y = [item["map"] for item in frames_eval]
    map_errs = np.array([item["map_err"] for item in frames_eval]).squeeze().T
    if map_errs.ndim == 1:
        map_errs = map_errs.reshape(2, 1)

    ax.errorbar(x, r1_y, yerr=r1_errs, fmt="-o", color="#1f77b4", lw=2.2, capsize=4, elinewidth=1.5, label="Rank-1 Accuracy (%)")
    ax.errorbar(x, map_y, yerr=map_errs, fmt="-s", color="#ff7f0e", lw=2.2, capsize=4, elinewidth=1.5, label="mAP (%)")

    ax.set_xscale("log", base=2)
    ax.set_xticks(x)
    ax.set_xticklabels([str(val) for val in x], fontsize=11)
    ax.set_xlabel("Number of Sampled Frames per Video Clip (Log2 Scale)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Performance Metric (%)", fontsize=12, fontweight="bold")
    ax.set_title("Temporal Scaling: Identification Accuracy vs. Number of Frames", fontsize=13, fontweight="bold", pad=12)
    ax.grid(True, which="both", ls="--", alpha=0.45)
    ax.legend(frameon=True, fontsize=11, loc="lower right")

    fig_dir = ROOT_DIR / "evaluation" / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_dir / "frame_length_scaling.png", dpi=300, bbox_inches="tight")
    plt.savefig(fig_dir / "frame_length_scaling.pdf", bbox_inches="tight")
    print("-> Saved figure to " + str(fig_dir / "frame_length_scaling.png"))
    plt.show()
else:
    print(f"-> [{len(frames_eval)}/6 frame points verified so far] Temporal scaling curve will be rendered once evaluation jobs complete.")


-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> Saved figure to /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553/evaluation/figures/frame_length_scaling.png


## 3. Feature Pooling & Aggregation Ablation (Temporal & Spatial)

Comparing aggregation strategies:
1. **Temporal Aggregation**: Learnable temporal self-attention across 8 video frames vs. simple temporal average pooling and element-wise max pooling.
2. **Spatial Token Pooling**: Global class token `[CLS]` vs. spatial patch mean pooling, generalized mean pooling (GeM, p=3.0), and combinations.

In [5]:
# =====================================================================
# 3. Feature Pooling & Aggregation: Temporal & Spatial Tables
# =====================================================================
ABLATIONS_DIR = ROOT_DIR / "evaluation" / "csvs_ablations"
CLOSED_DIR = ROOT_DIR / "evaluation" / "csvs_closed_results"

def get_pooling_row(name, path, fallback_path=None, ref_r1=None, ref_map=None, desc=""):
    p = path if path.exists() and path.stat().st_size > 0 else (fallback_path if fallback_path and fallback_path.exists() and fallback_path.stat().st_size > 0 else None)
    if not p or p.stat().st_mtime < BASE_MTIME:
        return {"Method": name, "Rank-1 (%) [95% CI]": "Pending...", "Rank-5 (%) [95% CI]": "Pending...", "mAP (%) [95% CI]": "Pending...", "Δ (R-1 / mAP)": "-", "Status": "In Progress", "Description": desc}
    res = bootstrap_from_csv(str(p), m=100, mode="closed")
    m_mean = res.get("mAP_boot_mean", 0.0) * 100
    m_std = res.get("mAP_std", 0.0) * 100
    m_ci = res.get("mAP_ci", (0.0, 0.0))
    cmc_boot = res.get("cmc_boot_mean", [0.0]*10)
    cmc_std = res.get("cmc_std", np.zeros(10))
    cmc_l = res.get("cmc_ci_lower", cmc_boot)
    cmc_u = res.get("cmc_ci_upper", cmc_boot)
    cmc = cmc_boot[0] * 100
    r1_std = cmc_std[0] * 100 if len(cmc_std) > 0 else 0.0
    cmc_ci = (cmc_l[0] * 100, cmc_u[0] * 100)
    r5 = cmc_boot[4] * 100 if len(cmc_boot) >= 5 else 0.0
    r5_std = cmc_std[4] * 100 if len(cmc_std) >= 5 else 0.0
    r5_ci = (cmc_l[4] * 100, cmc_u[4] * 100) if len(cmc_l) >= 5 else (r5, r5)
    delta = "Reference" if ref_r1 is None else f"{(cmc - ref_r1):+.1f} / {(m_mean - ref_map):+.1f}"
    return {
        "Method": name,
        "Rank-1 (%) [95% CI]": f"{cmc:.1f} ± {r1_std:.1f} [{cmc_ci[0]:.1f}-{cmc_ci[1]:.1f}]",
        "Rank-5 (%) [95% CI]": f"{r5:.1f} ± {r5_std:.1f} [{r5_ci[0]:.1f}-{r5_ci[1]:.1f}]",
        "mAP (%) [95% CI]": f"{m_mean:.1f} ± {m_std:.1f} [{m_ci[0]*100:.1f}-{m_ci[1]*100:.1f}]",
        "Δ (R-1 / mAP)": delta,
        "Status": "Verified",
        "Description": desc,
        "_r1": cmc, "_r5": r5, "_map": m_mean
    }

# 1. Temporal Pooling Across Frames
base_temp = get_pooling_row(
    "Temporal Attention Pooling (Baseline)",
    CLOSED_DIR / "baseline_vid2vid" / "closed_dist_matrix.csv",
    desc="Learnable temporal self-attention across 8 frames"
)
temp_r1, temp_map = base_temp.get("_r1"), base_temp.get("_map")

temporal_rows = [
    base_temp,
    get_pooling_row(
        "Temporal Mean Pooling",
        ABLATIONS_DIR / "temporal_mean_vid2vid" / "closed_dist_matrix.csv",
        ref_r1=temp_r1, ref_map=temp_map,
        desc="Global average pooling across 8 video frames"
    ),
    get_pooling_row(
        "Temporal Max Pooling",
        ABLATIONS_DIR / "temporal_max_vid2vid" / "closed_dist_matrix.csv",
        ref_r1=temp_r1, ref_map=temp_map,
        desc="Element-wise maximum across 8 video frames"
    )
]
df_temporal = pd.DataFrame(temporal_rows)[["Method", "Rank-1 (%) [95% CI]", "Rank-5 (%) [95% CI]", "mAP (%) [95% CI]", "Δ (R-1 / mAP)", "Status", "Description"]]

print("=== Temporal Aggregation (Across Video Frames) ===")
display(df_temporal)



-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
=== Temporal Aggregation (Across Video Frames) ===
                                  Method  ...                                        Description
0  Temporal Attention Pooling (Baseline)  ...  Learnable temporal self-attention across 8 frames
1                  Temporal Mean Pooling  ...       Global average pooling across 8 video frames
2                   Temporal Max Pooling  ...         Element-wise maximum across 8 video frames

[3 rows x 7 columns]


## 4. Gallery Size Sensitivity & Failure Analysis

> [!NOTE]
> The detailed **Gallery Size Sensitivity Analysis** (both total video volume scaling and distractor identity scaling), along with the **Best- vs. Worst-Performing Identity Analysis** and **Inter-Identity Confusion Analysis**, have been transferred to the dedicated evaluation notebook: [evaluate_identity_and_confusion.ipynb](file:///d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553/evaluation/evaluate_identity_and_confusion.ipynb).
